In [0]:
import requests
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import  time

# Initialize Spark Session
spark = SparkSession.builder.appName("Weather Data Extraction").getOrCreate()

# OpenWeather API Configuration
base_url = "https://api.openweathermap.org/data/2.5/weather"
api_key = "your_openweather_api_key"  # Replace with your OpenWeather API key

# Function to fetch weather data for a city
def fetch_weather_data(city):
    params = {
        "q": city,
        "appid": api_key,
        "units": "metric"  # Get temperature in Celsius
    }
    response = requests.get(base_url, params=params)
    response.raise_for_status()  # Raise exception for HTTP errors
    return response.json()

# Function to process data for multiple cities
def fetch_weather_for_cities(cities):
    all_data = []
    for city in cities:
        print(f"Fetching weather data for: {city}")
        try:
            data = fetch_weather_data(city)
            all_data.append(data)
        except Exception as e:
            print(f"Error fetching data for {city}: {e}")
        time.sleep(1)  # Respect API rate limits
    return all_data

# List of cities to fetch weather data for
cities = ["New York", "London", "Tokyo", "Mumbai", "Sydney"]

# Fetch weather data
weather_data = fetch_weather_for_cities(cities)

# Convert data to Spark DataFrame
rdd = spark.sparkContext.parallelize(weather_data)
df = spark.read.json(rdd)

# Show the DataFrame
df.show(truncate=False)

# Save the DataFrame to Delta Lake for further processing
output_path = "/mnt/delta/weather_data"
df.write.format("delta").mode("overwrite").save(output_path)

print(f"Data saved to Delta Lake at: {output_path}")
